#  **Scaling E-Scooter Battery Collection to 1000+ Scooters with Google's OR-Tools**

---

## 📋 What This Notebook Does

This notebook implements a **production-ready CVRP solver** using Google's OR-Tools framework. Unlike brute force enumeration, this approach scales to real-world problems by using advanced algorithms developed over 20+ years of research.

### 🎯 **The OR-Tools Advantage**
- **Same Math**: Uses our exact MILP formulation with binary variables x^k_{ij}
- **Smart Solving**: Constraint propagation + local search heuristics
- **Real Scale**: Handles 1000 scooters in 30 seconds vs. 2000 years for brute force

### 🔧 **What You'll See**
1. How OR-Tools translates our mathematical constraints into code
2. Advanced search strategies (PATH_CHEAPEST_ARC + GUIDED_LOCAL_SEARCH)
3. Scalability from 8 scooters to 1000+ scooters
4. Beautiful Manhattan distance routing visualization

##  1: Imports and Setup
**Import OR-Tools and essential libraries for our production-grade CVRP solver.**

In [13]:
import random
import numpy as np
import matplotlib.pyplot as plt
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import time

##  2: Problem Parameters
**Configure your CVRP instance - easily switch between small demo and large-scale problems.**


In [ ]:
# CUSTOMIZE YOUR PARAMETERS HERE
# Small demo (matches presentation slides)
N_SCOOTERS = 8      # Number of scooters to collect
N_TRUCKS = 2        # Number of trucks available
CAPACITY = 4        # Max scooters per truck
GRID_SIZE = 15      # Grid size (GRID_SIZE x GRID_SIZE)
SEED = None           # Random seed for reproducibility (replace by a number)

# # Uncomment for LARGE SCALE demo (1000 scooters!)
# N_SCOOTERS = 1000
# N_TRUCKS = 20
# CAPACITY = 50
# GRID_SIZE = 100
# SEED = None

print(" Battery Collection CVRP Solver")
print("=" * 40)
print(f" CVRP Problem: {N_SCOOTERS} scooters, {N_TRUCKS} trucks, capacity {CAPACITY}")
print(f" Grid: {GRID_SIZE}x{GRID_SIZE}, Binary variables: {(N_SCOOTERS+1)**2 * N_TRUCKS:,}")

 ## 3: Feasibility Check and Location Generation
**Verify our problem is solvable and generate random scooter locations on the grid.**

In [ ]:
# Feasibility check
if N_SCOOTERS > N_TRUCKS * CAPACITY:
    print(f" INFEASIBLE: Need {N_SCOOTERS} scooters but only have {N_TRUCKS * CAPACITY} capacity")
    raise ValueError("Problem is infeasible - not enough truck capacity!")

print("✅ Problem is feasible!")

# Generate locations
random.seed(SEED)
depot = (GRID_SIZE // 2, GRID_SIZE // 2)
scooters = [(random.randint(0, GRID_SIZE), random.randint(0, GRID_SIZE)) for _ in range(N_SCOOTERS)]
coords = [depot] + scooters
n_locations = len(coords)

print(f" Depot location: {depot}")
print(f" Generated {N_SCOOTERS} random scooter locations")
print(f" Total locations to visit: {n_locations}")

##  4: Distance Matrix Calculation
**Calculate Manhattan distances between all locations - this becomes our cost matrix.**

In [ ]:
# Distance matrix (Manhattan distance for grid-based routing)
distance_matrix = np.zeros((n_locations, n_locations), dtype=int)
for i, (x1, y1) in enumerate(coords):
    for j, (x2, y2) in enumerate(coords):
        distance_matrix[i, j] = abs(x1 - x2) + abs(y1 - y2)


print("  Distance matrix calculated!")
print(f"   Matrix size: {distance_matrix.shape}")
print(f"   Max distance: {distance_matrix.max()}")
print(f"   Avg distance: {distance_matrix[distance_matrix > 0].mean():.2f}")


# Show sample distances for small problems
if N_SCOOTERS <= 8:
    print(f"\n Sample distances from depot:")
    for i in range(1, min(6, n_locations)):
        print(f"   Depot → Scooter {i}: {distance_matrix[0, i]}")

##  5: OR-Tools Model Setup
**Create the routing model and translate our mathematical objective into OR-Tools language.**

In [ ]:
# OR-Tools setup - this handles our binary variables x^k_ij automatically
manager = pywrapcp.RoutingIndexManager(n_locations, N_TRUCKS, 0)
routing = pywrapcp.RoutingModel(manager)


# Distance callback - implements our objective function min ∑∑∑ d_ij × x^k_ij
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)


print(" OR-Tools routing model created!")
print(f"   Managing {n_locations} locations and {N_TRUCKS} trucks")
print(" Objective function registered (minimize total distance)")

##  6: Constraint Implementation
**Add capacity constraints - each truck can visit at most Q scooters.**

In [ ]:
# Capacity constraint - implements ∑∑ x^k_ij ≤ Q for each truck k
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return 0 if from_node == 0 else 1  # Depot=0 demand, Scooter=1 demand

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index, 
    0,                          # No slack 
    [CAPACITY] * N_TRUCKS,      # Capacity for each truck
    True,                       # Start cumul to zero
    "Capacity"
)

print(" Capacity constraints added!")
print(f"   Each truck limited to {CAPACITY} scooters")
print(" Coverage and flow constraints handled automatically by OR-Tools")

##  7: Advanced Search Configuration
**Configure the search strategy - this is where 20+ years of Google research make a different!**

In [ ]:
# Search parameters - advanced algorithms for real-world performance
search_parameters = pywrapcp.DefaultRoutingSearchParameters()

# First solution strategy: PATH_CHEAPEST_ARC
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC

# Local search metaheuristic: GUIDED_LOCAL_SEARCH  
search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH

# Time limits based on problem size
time_limit = 30 if N_SCOOTERS > 100 else 10
search_parameters.time_limit.FromSeconds(time_limit)

print(" Advanced search strategy configured!")
print(f"   Strategy: PATH_CHEAPEST_ARC + GUIDED_LOCAL_SEARCH")
print(f"   Time limit: {time_limit} seconds")
print(" Ready to solve!")

##  8: Solve the Problem
**Run the OR-Tools solver and measure performance.**

In [ ]:
# Solve the problem!
print("🚀 Solving...")
start_time = time.time()
solution = routing.SolveWithParameters(search_parameters)
solve_time = time.time() - start_time

if not solution:
    print(" No solution found")
    raise RuntimeError("Solver failed to find a solution")
else:
    print(" SOLUTION FOUND!")
    print(f"⏱  Solve Time: {solve_time:.3f}s")

##  9: Extract and Analyze Solution
**Parse the OR-Tools solution and extract route information.**

In [ ]:
# Extract solution details
total_cost = 0
routes = []
scooters_collected = set()

# iterate over the trucks to find the best route for each truck
for truck_id in range(N_TRUCKS):
    route = []
    route_distance = 0
    index = routing.Start(truck_id)
    
    # while not at the end of the route, add the node to the route and collect the scooter if it's not the depot
    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(node)
        if node != 0:  # Not depot
            scooters_collected.add(node)
        
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        if not routing.IsEnd(index):
            route_distance += distance_callback(previous_index, index)
    
    route.append(0)  # Return to depot
    route_distance += distance_callback(previous_index, routing.End(truck_id))
    
    routes.append(route)
    total_cost += route_distance

trucks_used = sum(1 for route in routes if len(route) > 2)

print(f" SOLUTION ANALYSIS:")
print(f"   Total Distance: {total_cost:,}")
print(f"   Trucks Used: {trucks_used}/{N_TRUCKS}")
print(f"   Scooters Collected: {len(scooters_collected)}/{N_SCOOTERS}")
print(f"   Efficiency: {total_cost/len(scooters_collected):.1f} distance per scooter")

##  10: Display Routes
**Show detailed route information for each truck.**

In [ ]:
# Show routes for small problems
print(f"\n ROUTE DETAILS:")
if N_SCOOTERS <= 20:
    for i, route in enumerate(routes):
        if len(route) > 2:
            scooter_count = len([node for node in route if node != 0])
            route_distance = sum(distance_matrix[route[j]][route[j+1]] for j in range(len(route)-1))
            print(f"   Truck {i+1}: {route}")
            print(f"   ├─ Load: {scooter_count}/{CAPACITY} scooters")
            print(f"   └─ Distance: {route_distance}")
else:
    print(f"   📋 Problem too large to show individual routes")
    print(f"   📊 Use visualization below to see the solution")

##  11: Visualization
**Create a beautiful plot showing the Manhattan routing solution.**

In [ ]:
def plot_solution(coords, routes, total_cost, solve_time, n_trucks, grid_size):
    """Create visualization of the OR-Tools solution"""
    plt.figure(figsize=(12, 10))
    
    # Generate colors for all trucks
    truck_colors = plt.cm.rainbow(np.linspace(0, 1, n_trucks))
    
    # Set axis limits first to avoid warnings
    plt.xlim(-0.5, grid_size + 0.5)
    plt.ylim(-0.5, grid_size + 0.5)
    
    # Grid lines for Manhattan distance visualization
    for x in range(grid_size + 1):
        plt.axvline(x=x, color='#e0e0e0', linewidth=0.5, zorder=1)
    for y in range(grid_size + 1):
        plt.axhline(y=y, color='#e0e0e0', linewidth=0.5, zorder=1)
    
    # Depot
    depot_x, depot_y = coords[0]
    plt.scatter(depot_x, depot_y, c='black', s=200, marker='s', 
               label='Depot', zorder=5, edgecolors='white', linewidth=2)
    
    # Plot routes with Manhattan paths
    for truck_id, route in enumerate(routes):
        if len(route) <= 2:
            continue
            
        color = truck_colors[truck_id]
        
        # Draw Manhattan path segments (horizontal then vertical)
        path_points = []
        for i in range(len(route) - 1):
            start = coords[route[i]]
            end = coords[route[i + 1]]
            
            # Manhattan routing: go horizontal first, then vertical
            intermediate = (end[0], start[1])
            
            if i == 0:  # First segment
                path_points.extend([start, intermediate, end])
            else:
                path_points.extend([intermediate, end])
        
        # Plot the complete path
        if len(path_points) >= 2:
            xs, ys = zip(*path_points)
            plt.plot(xs, ys, color=color, linewidth=3, 
                    label=f'Truck {truck_id + 1}', alpha=0.8, zorder=3)
        
        # Plot scooters visited by this truck
        for node in route[1:-1]:  # Exclude depot
            x, y = coords[node]
            plt.scatter(x, y, color=color, s=80, marker='o', 
                       edgecolors='black', linewidth=1, zorder=4)
    
    plt.title(f'OR-Tools CVRP Solution\nDistance: {total_cost:,} | Time: {solve_time:.3f}s | Trucks: {trucks_used}', 
              fontsize=14, fontweight='bold')
    plt.xlabel('X Coordinate', fontsize=12)
    plt.ylabel('Y Coordinate', fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

# Create the visualization
plot_solution(coords, routes, total_cost, solve_time, N_TRUCKS, GRID_SIZE)

##  12: Performance Summary
**Final performance metrics and scaling analysis.**

In [ ]:
print(" FINAL PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Problem Size:")
print(f"  • {N_SCOOTERS:,} scooters")
print(f"  • {N_TRUCKS} trucks (capacity {CAPACITY} each)")
print(f"  • {(N_SCOOTERS+1)**2 * N_TRUCKS:,} binary variables")

print(f"\nSolution Quality:")
print(f"  • Total distance: {total_cost:,}")
print(f"  • Trucks used: {trucks_used}/{N_TRUCKS}")
print(f"  • Efficiency: {total_cost/len(scooters_collected):.1f} distance/scooter")

print(f"\nPerformance:")
print(f"  • Solve time: {solve_time:.3f} seconds")
print(f"  • Algorithm: OR-Tools with advanced search")


